# Project: Time Series & Category Charts
### Data Science & Analytics Portfolio

**Objective:**
1. **Time Series Analysis:** Plot daily sales over time (line charts) along with monthly and quarterly aggregations.
2. **Category Comparisons:** Compare product category revenues and average order values using bar charts.
3. **Proportional Share:** Visualize market and category share using pie and donut charts.
4. **Export Artifacts:** Automatically save all charts to high-resolution PNGs and export executive summary reports (TXT and CSV).
5. **Technical Discussion:** In-depth evaluation of chart selection principles and formatting standards (labels, legends, axes).

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns
from datetime import datetime

# Set global plotting aesthetic style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial', 'DejaVu Sans', 'Helvetica'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Create directory for exported PNG charts and reports
output_dir = 'output_charts'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: '{output_dir}'")

## 2. Dataset Synthesis: Multi-Year Sales Transactions
We generate realistic daily transaction records across 5 retail product categories over a 2-year horizon (2024–2025) featuring:
- Day-of-week seasonality (weekend shopping lift)
- Monthly seasonality (Q4 holiday surge)
- Long-term upward business growth trend

In [ ]:
def generate_sales_data(start_date="2024-01-01", end_date="2025-12-31", random_seed=42):
    np.random.seed(random_seed)
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    
    categories = ['Electronics', 'Furniture', 'Office Supplies', 'Apparel', 'Home Appliances']
    base_prices = {
        'Electronics': 250.0,
        'Furniture': 180.0,
        'Office Supplies': 35.0,
        'Apparel': 65.0,
        'Home Appliances': 140.0
    }
    
    records = []
    
    for current_date in date_range:
        # Day-of-week effect: higher sales on weekends
        day_factor = 1.25 if current_date.weekday() >= 5 else 0.95
        # Monthly seasonality: higher sales in Q4 (Nov/Dec)
        month_factor = 1.35 if current_date.month in [11, 12] else (1.1 if current_date.month in [6, 7] else 1.0)
        # Growth trend across 2 years
        trend_factor = 1.0 + (current_date - pd.to_datetime(start_date)).days / 1200.0
        
        for category in categories:
            base_orders = {
                'Electronics': 12,
                'Furniture': 8,
                'Office Supplies': 35,
                'Apparel': 25,
                'Home Appliances': 10
            }[category]
            
            lambda_orders = max(1, base_orders * day_factor * month_factor * trend_factor)
            order_count = np.random.poisson(lambda_orders)
            unit_price = max(np.random.normal(base_prices[category], base_prices[category] * 0.1), 5.0)
            revenue = round(order_count * unit_price, 2)
            
            records.append({
                'Date': current_date,
                'Category': category,
                'Orders': order_count,
                'Revenue': revenue
            })
            
    df = pd.DataFrame(records)
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.to_period('M')
    df['Quarter'] = df['Date'].dt.to_period('Q')
    return df

df = generate_sales_data()
print(f"Generated {len(df):,} records spanning {df['Date'].min().date()} to {df['Date'].max().date()}.")
df.head()

## 3. Helper Formatter Function
Custom axis formatter to convert large numeric values into clean `$K` and `$M` currencies.

In [ ]:
def currency_formatter(x, pos):
    if x >= 1e6:
        return f"${x * 1e-6:.1f}M"
    elif x >= 1e3:
        return f"${x * 1e-3:.0f}K"
    else:
        return f"${x:.0f}"

## 4. Time Series Charts: Sales Over Time

### 4.1 Daily Revenue with 30-Day Moving Average Line Chart

In [ ]:
daily_sales = df.groupby('Date')['Revenue'].sum().reset_index()
daily_sales['MA_30'] = daily_sales['Revenue'].rolling(window=30, min_periods=7).mean()

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)

# Raw daily revenue line (muted / semi-transparent)
ax.plot(daily_sales['Date'], daily_sales['Revenue'], color='#90caf9', alpha=0.6, linewidth=1.2, label='Daily Revenue')

# 30-day moving average trend line
ax.plot(daily_sales['Date'], daily_sales['MA_30'], color='#0d47a1', linewidth=2.4, label='30-Day Rolling Average Trend')

# Styling and formatting
ax.set_title('Overall Daily Sales Over Time (2024 - 2025)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=11, fontweight='medium', labelpad=10)
ax.set_ylabel('Total Revenue (USD)', fontsize=11, fontweight='medium', labelpad=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor='white', edgecolor='#e0e0e0', loc='upper left')

plt.tight_layout()
chart_1_path = os.path.join(output_dir, "01_sales_over_time_daily.png")
plt.savefig(chart_1_path, dpi=300)
plt.show()
print(f"Saved: {chart_1_path}")

### 4.2 Monthly Aggregation: Total & Category Breakdown

In [ ]:
monthly_cat = df.groupby(['Month', 'Category'])['Revenue'].sum().reset_index()
monthly_cat['Month_Date'] = monthly_cat['Month'].dt.to_timestamp()

monthly_total = df.groupby('Month')['Revenue'].sum().reset_index()
monthly_total['Month_Date'] = monthly_total['Month'].dt.to_timestamp()

palette = sns.color_palette("tab10", n_colors=df['Category'].nunique())
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9.5), sharex=True, dpi=300)

# Top Subplot: Total Monthly Revenue
ax1.plot(monthly_total['Month_Date'], monthly_total['Revenue'], marker='o', color='#1b5e20', linewidth=2.5, markersize=5, label='Total Monthly Revenue')
ax1.fill_between(monthly_total['Month_Date'], monthly_total['Revenue'], color='#a5d6a7', alpha=0.35)
ax1.set_title('Monthly Total Sales Aggregation', fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('Total Revenue', fontsize=11, fontweight='medium')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper left', frameon=True)

# Direct value callouts on monthly points
for _, row in monthly_total.iterrows():
    ax1.annotate(f"${row['Revenue']/1000:.0f}K", xy=(row['Month_Date'], row['Revenue']),
                 xytext=(0, 6), textcoords='offset points', ha='center', fontsize=7.5, color='#1b5e20', fontweight='semibold')

# Bottom Subplot: Category Monthly Trends
for idx, cat in enumerate(df['Category'].unique()):
    sub_data = monthly_cat[monthly_cat['Category'] == cat]
    ax2.plot(sub_data['Month_Date'], sub_data['Revenue'], marker='s', markersize=4, linewidth=1.8, color=palette[idx], label=cat)

ax2.set_title('Monthly Revenue Breakdown by Category', fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel('Month', fontsize=11, fontweight='medium', labelpad=10)
ax2.set_ylabel('Category Revenue', fontsize=11, fontweight='medium')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=40, ha='right')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

plt.tight_layout()
chart_2_path = os.path.join(output_dir, "02_monthly_sales_aggregation.png")
plt.savefig(chart_2_path, dpi=300)
plt.show()
print(f"Saved: {chart_2_path}")

### 4.3 Quarterly Aggregation: Stacked Composition & Grouped Bar Charts

In [ ]:
quarterly = df.groupby(['Quarter', 'Category'])['Revenue'].sum().unstack()
quarter_labels = [str(q) for q in quarterly.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.8), dpi=300)

# Stacked Quarterly Composition
quarterly.plot(kind='bar', stacked=True, ax=ax1, colormap='Blues_r', edgecolor='white', linewidth=0.8, width=0.65)
ax1.set_title('Quarterly Revenue (Composition by Category)', fontsize=12.5, fontweight='bold', pad=12)
ax1.set_xlabel('Fiscal Quarter', fontsize=11, fontweight='medium', labelpad=8)
ax1.set_ylabel('Total Revenue (USD)', fontsize=11, fontweight='medium')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax1.set_xticklabels(quarter_labels, rotation=45, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.6)
ax1.legend(title='Category', loc='upper left', frameon=True, fontsize=8.5)

# Grouped Quarterly Comparison
quarterly.plot(kind='bar', stacked=False, ax=ax2, colormap='viridis', edgecolor='none', width=0.75)
ax2.set_title('Quarterly Revenue (Grouped Category Comparison)', fontsize=12.5, fontweight='bold', pad=12)
ax2.set_xlabel('Fiscal Quarter', fontsize=11, fontweight='medium', labelpad=8)
ax2.set_ylabel('Quarterly Revenue (USD)', fontsize=11, fontweight='medium')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax2.set_xticklabels(quarter_labels, rotation=45, ha='right')
ax2.grid(axis='y', linestyle='--', alpha=0.6)
ax2.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, fontsize=8.5)

plt.tight_layout()
chart_3_path = os.path.join(output_dir, "03_quarterly_sales_aggregation.png")
plt.savefig(chart_3_path, dpi=300)
plt.show()
print(f"Saved: {chart_3_path}")

## 5. Category Comparisons: Bar Charts
- **Horizontal Bar:** Ranked comparison of total category revenue.
- **Vertical Bar:** Comparison of Average Order Value (AOV) across product categories.

In [ ]:
cat_summary = df.groupby('Category').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Orders=('Orders', 'sum'),
    Avg_Order_Value=('Revenue', lambda x: x.sum() / df.loc[x.index, 'Orders'].sum())
).sort_values(by='Total_Revenue', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.8), dpi=300)

# Horizontal Bar: Total Revenue
colors = sns.color_palette("crest", len(cat_summary))
bars1 = ax1.barh(cat_summary.index, cat_summary['Total_Revenue'], color=colors, edgecolor='#333333', linewidth=0.5, height=0.6)
ax1.set_title('Total Revenue by Category (Ranking)', fontsize=12.5, fontweight='bold', pad=12)
ax1.set_xlabel('Total Revenue (USD)', fontsize=11, fontweight='medium')
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax1.grid(axis='x', linestyle='--', alpha=0.6)

# Value labels at bar tips
for bar in bars1:
    width = bar.get_width()
    ax1.text(width * 1.01, bar.get_y() + bar.get_height() / 2, f"${width:,.0f}",
             va='center', ha='left', fontsize=9, fontweight='bold', color='#212121')
ax1.set_xlim(0, cat_summary['Total_Revenue'].max() * 1.18)

# Vertical Bar: Average Order Value
bars2 = ax2.bar(cat_summary.index, cat_summary['Avg_Order_Value'], color='#ff8a65', edgecolor='#bf360c', linewidth=0.8, width=0.55)
ax2.set_title('Average Order Value (AOV) by Category', fontsize=12.5, fontweight='bold', pad=12)
ax2.set_xlabel('Product Category', fontsize=11, fontweight='medium', labelpad=8)
ax2.set_ylabel('Average Order Value ($)', fontsize=11, fontweight='medium')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(currency_formatter))
ax2.set_xticklabels(cat_summary.index, rotation=30, ha='right')
ax2.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, height + 3, f"${height:.2f}",
             va='bottom', ha='center', fontsize=9, fontweight='bold', color='#bf360c')
ax2.set_ylim(0, cat_summary['Avg_Order_Value'].max() * 1.15)

plt.tight_layout()
chart_4_path = os.path.join(output_dir, "04_category_comparison_bar_charts.png")
plt.savefig(chart_4_path, dpi=300)
plt.show()
print(f"Saved: {chart_4_path}")

## 6. Category Share: Pie & Modern Donut Charts

In [ ]:
cat_revenue = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
explode = (0.05, 0, 0, 0, 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5), dpi=300)

# Classic Pie Chart
wedges1, texts1, autotexts1 = ax1.pie(
    cat_revenue, explode=explode, labels=cat_revenue.index, autopct='%1.1f%%',
    pctdistance=0.75, startangle=140, colors=colors,
    wedgeprops=dict(width=1.0, edgecolor='white', linewidth=2)
)
plt.setp(autotexts1, size=9.5, weight="bold", color="white")
plt.setp(texts1, size=10, weight="medium")
ax1.set_title('Category Revenue Share\n(Standard Pie Chart)', fontsize=12.5, fontweight='bold', pad=15)

# Modern Donut Chart with KPI in Center
wedges2, texts2, autotexts2 = ax2.pie(
    cat_revenue, explode=explode, autopct='%1.1f%%',
    pctdistance=0.8, startangle=140, colors=colors,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2)
)
plt.setp(autotexts2, size=9.5, weight="bold")

total_rev = cat_revenue.sum()
ax2.text(0, 0, f"Total Revenue\n${total_rev:,.0f}", ha='center', va='center', fontsize=11, fontweight='bold', color='#333333')
ax2.set_title('Proportional Revenue Share\n(Modern Donut Chart)', fontsize=12.5, fontweight='bold', pad=15)

ax2.legend(
    wedges2,
    [f"{cat} (${val:,.0f})" for cat, val in cat_revenue.items()],
    title="Categories (Total)",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    frameon=True
)

plt.tight_layout()
chart_5_path = os.path.join(output_dir, "05_category_revenue_share_pie_donut.png")
plt.savefig(chart_5_path, dpi=300)
plt.show()
print(f"Saved: {chart_5_path}")

## 7. Exporting Summary Reports (TXT & CSV)

In [ ]:
total_revenue = df['Revenue'].sum()
total_orders = df['Orders'].sum()
avg_order_value = total_revenue / total_orders if total_orders else 0
date_min = df['Date'].min().strftime('%Y-%m-%d')
date_max = df['Date'].max().strftime('%Y-%m-%d')

# Category aggregation
cat_stats = df.groupby('Category').agg(
    Total_Revenue=('Revenue', 'sum'),
    Order_Count=('Orders', 'sum'),
    Avg_Order_Value=('Revenue', lambda x: x.sum() / df.loc[x.index, 'Orders'].sum())
).reset_index()
cat_stats['Revenue_Share_Pct'] = (cat_stats['Total_Revenue'] / total_revenue) * 100
cat_stats = cat_stats.sort_values(by='Total_Revenue', ascending=False)

# Quarterly aggregation
quarterly_stats = df.groupby('Quarter').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Orders=('Orders', 'sum')
).reset_index()
quarterly_stats['Quarter'] = quarterly_stats['Quarter'].astype(str)

# Export CSV files
cat_stats.to_csv(os.path.join(output_dir, "category_sales_summary.csv"), index=False)
quarterly_stats.to_csv(os.path.join(output_dir, "quarterly_sales_summary.csv"), index=False)

# Generate executive text report
summary_text = f"""================================================================================
EXECUTIVE SALES & REVENUE PERFORMANCE SUMMARY REPORT
Project: Time Series & Category Charts
Date Range Analyzed: {date_min} to {date_max}
================================================================================

1. KEY PERFORMANCE INDICATORS (KPIs):
- Total Gross Revenue:        ${total_revenue:,.2f}
- Total Orders Processed:     {total_orders:,} units
- Average Order Value (AOV):  ${avg_order_value:.2f}
- Top Category by Revenue:    {cat_stats.iloc[0]['Category']} (${cat_stats.iloc[0]['Total_Revenue']:,.2f} / {cat_stats.iloc[0]['Revenue_Share_Pct']:.1f}%)

2. CATEGORY BREAKDOWN:
{cat_stats.to_string(index=False, formatters={
    'Total_Revenue': lambda x: f"${x:,.2f}",
    'Avg_Order_Value': lambda x: f"${x:.2f}",
    'Revenue_Share_Pct': lambda x: f"{x:.1f}%"
})}

3. QUARTERLY RUN-RATE:
{quarterly_stats.to_string(index=False, formatters={
    'Total_Revenue': lambda x: f"${x:,.2f}",
    'Total_Orders': lambda x: f"{x:,}"
})}

4. EXPORTED HIGH-RESOLUTION CHARTS (PNG):
- 01_sales_over_time_daily.png
- 02_monthly_sales_aggregation.png
- 03_quarterly_sales_aggregation.png
- 04_category_comparison_bar_charts.png
- 05_category_revenue_share_pie_donut.png
================================================================================
"""

report_path = os.path.join(output_dir, "sales_summary_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print(f"Executive report saved to: '{report_path}'")

## 8. Technical Discussion & Design Rationale

### 8.1 Chart Choice Principles
1. **Line Charts (Time Series Analysis):**
   - **Strengths:** Excellent for continuous chronological series. The human eye intuitively interprets lines as movement through time, revealing seasonality, trends, and inflections.
   - **Aggregations:** Daily data captures granular variance (e.g. weekend lifts), while rolling moving averages and monthly/quarterly aggregations remove high-frequency noise.

2. **Bar Charts (Categorical Comparisons):**
   - **Strengths:** Human perception assesses differences in length along an aligned baseline with superior accuracy compared to angles or areas.
   - **Orientation:** Horizontal bars (`barh`) are used for category revenue rankings because category names can be displayed horizontally without awkward tilting.
   - **Grouped vs. Stacked Bars:** Grouped bars compare categories across quarters; stacked bars show both the macro quarterly total and category contribution.

3. **Pie vs. Donut Charts (Proportions):**
   - **Considerations:** Pie charts should only be used when displaying parts of a single whole ($100\%$ total) and limited to $\le 5$ categories.
   - **Advantage of Donut Charts:** Removing the center vertex reduces visual distortion and creates space to display key summary statistics (e.g., total gross revenue).

### 8.2 Label, Legend & Axis Formatting Standards
- **Axis Scaling:** Currency values are formatted with readable unit suffixes (`$K`, `$M`) to avoid raw integer clutter.
- **Date Locators:** `mdates.MonthLocator` and `mdates.DateFormatter('%b %Y')` ensure tidy tick intervals.
- **Baseline Zero:** Bar charts always start at zero ($0$) to prevent misleading visual exaggeration.
- **Legend Placement:** Placed outside data plotting areas (`bbox_to_anchor=(1.02, 1)`) to avoid obscuring data points.